# 7.9um vs 2.4um correlation analysis — scroll4 w023

**Question**: does the 7.9um/54keV scan of scroll4 w023 contain *any* voxel-level signal that correlates with what the 2.4um ink model found?

Method:
1. Warp the continuous 2.4um ink probability tif into the 7.9um frame (same validated warp, IoU 0.71).
2. Load 7.9um voxels in the same region.
3. Visually overlay: do ink-highlighted pixels in the 2.4um output visually correspond to anything in the 7.9um slice?
4. Extract per-tile features from the 7.9um scan and compute Spearman correlations against the 2.4um ink probability.
5. Check spatial consistency (does correlation hold across different sub-regions, or is it a confound?).

**Decision threshold**: r > 0.15 consistently across spatial quadrants = real signal. r ≤ 0.05 or inconsistent sign = dead end.

In [ ]:
import sys, os
sys.path.insert(0, '/vesuvius')

import numpy as np
import cv2
import zarr
import tifffile
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import spearmanr
from scipy.interpolate import RBFInterpolator
from warp_from_dots import detect_dots, match_by_color

# paths
Z79_PATH = '/vesuvius/ves_zarrs2/20240304161941.zarr'  # 7.9um w023 (h-flipped)
INK_TIF  = '/vesuvius/_ves_tmp/w023_ink_full.tif'      # 2.4um continuous ink prob (0-246)
SRC_DOTS = '/vesuvius/warp_MARK_2p4_source_dots.png'
DST_DOTS = '/vesuvius/warp_MARK_7p9_target_dots.png'
MASK_79  = '/vesuvius/masks/20240304161941.png'

H79, W79 = 13303, 31674
H24, W24 = 41860, 102360

# overlap region: right 30% x, top 40% y (validated ink-rich area)
cx0, cx1 = int(0.70 * W79), W79   # x[22172:31674]
cy0, cy1 = 0, int(0.40 * H79)    # y[0:5321]
cw, ch = cx1 - cx0, cy1 - cy0
print(f'7.9um crop: x[{cx0}:{cx1}] y[{cy0}:{cy1}]  ({cw} x {ch} px)')
print(f'ink tif   : {H24}x{W24}')

In [ ]:
# replicate warp_from_dots.py EXACTLY (ww=3600, smoothing=1.0, 4 corner anchors)
# this is the same recipe that produced the validated IoU=0.71 label alignment
def fit_warp_exact(src_dots_path, dst_dots_path, ww=3600):
    sdots, (sh, sw) = detect_dots(src_dots_path)
    ddots, (dh, dw) = detect_dots(dst_dots_path)
    pairs = match_by_color(sdots, ddots)
    ssc, dsc = ww / sw, ww / dw
    H24h = int(round(sh * ssc))
    H79h = int(round(dh * dsc))
    src, dst = [], []
    for xs, ys, xd, yd, _ in pairs:
        src.append([xs * ssc, ys * ssc])
        dst.append([xd * dsc, yd * dsc])
    for cx in (0, ww - 1):
        src += [[cx, 0], [cx, H24h - 1]]
        dst += [[cx, 0], [cx, H79h - 1]]
    src, dst = np.array(src, float), np.array(dst, float)
    fxp = RBFInterpolator(dst, src[:, 0], kernel='thin_plate_spline', smoothing=1.0)
    fyp = RBFInterpolator(dst, src[:, 1], kernel='thin_plate_spline', smoothing=1.0)
    # returned functions: accept (N,2) normalized 7.9 (u,v) -> normalized 2.4 coordinate
    def fx(pts):
        pts = np.atleast_2d(pts)
        return fxp(np.column_stack([pts[:, 0] * ww, pts[:, 1] * H79h])) / ww
    def fy(pts):
        pts = np.atleast_2d(pts)
        return fyp(np.column_stack([pts[:, 0] * ww, pts[:, 1] * H79h])) / H24h
    print(f'warp: {len(pairs)} pairs + 4 corners (ww={ww}, smoothing=1.0)')
    return fx, fy

fx, fy = fit_warp_exact(SRC_DOTS, DST_DOTS)

In [ ]:
# compute pixel maps for the 7.9 crop at NATIVE 7.9 resolution (1 output px = 1 7.9um px)
# coarse grid, then bilinear upsample (TPS is smooth so this is sub-pixel accurate)
step = 32
nc = max(2, cw // step + 2)
nr = max(2, ch // step + 2)
gc = np.linspace(0, cw - 1, nc)
gr = np.linspace(0, ch - 1, nr)
uu, vv = np.meshgrid((cx0 + gc) / W79, (cy0 + gr) / H79)
pts = np.column_stack([uu.ravel(), vv.ravel()])
su = (fx(pts).reshape(nr, nc) * W24).astype(np.float32)
sv = (fy(pts).reshape(nr, nc) * H24).astype(np.float32)
MAPX = cv2.resize(su, (cw, ch), interpolation=cv2.INTER_LINEAR)   # (ch, cw)
MAPY = cv2.resize(sv, (cw, ch), interpolation=cv2.INTER_LINEAR)

# warp the 2.4um continuous ink tif into 7.9 frame at native resolution
print('loading ink tif...')
ink = tifffile.imread(INK_TIF).astype(np.float32)   # (41860, 102360)
ix0 = max(0, int(MAPX.min()))
ix1 = min(W24, int(MAPX.max()) + 1)
iy0 = max(0, int(MAPY.min()))
iy1 = min(H24, int(MAPY.max()) + 1)
ink_sub = ink[iy0:iy1, ix0:ix1]
warped_ink = cv2.remap(ink_sub,
                       (MAPX - ix0).astype(np.float32),
                       (MAPY - iy0).astype(np.float32),
                       cv2.INTER_LINEAR, borderValue=0)
# warped_ink: (ch, cw) float32, 0-246, in native 7.9um pixel space
del ink, ink_sub  # free ~800MB
print(f'warped ink: {warped_ink.shape}, '
      f'range [{warped_ink.min():.0f}, {warped_ink.max():.0f}], '
      f'ink>60 frac: {(warped_ink > 60).mean():.3f}, '
      f'ink>120 frac: {(warped_ink > 120).mean():.3f}')

In [ ]:
# load 7.9um crop for multiple depth bands
# memory: 12 slices * 5321 * 9502 * 4 bytes ≈ 2.4 GB per band — fine
z79 = zarr.open(Z79_PATH, mode='r')
mask_79 = cv2.imread(MASK_79, 0)  # (13303, 31674)
mask_crop = mask_79[cy0:cy1, cx0:cx1]

print('loading 7.9um z28:40 (standard ink band)...')
blk_2840 = np.asarray(z79[28:40, cy0:cy1, cx0:cx1]).astype(np.float32)  # (12,ch,cw)
print('loading 7.9um z48:58 (alt band from depth profile)...')
blk_4858 = np.asarray(z79[48:58, cy0:cy1, cx0:cx1]).astype(np.float32)  # (10,ch,cw)
print('done')

mean_2840 = blk_2840.mean(0)   # (ch, cw)
std_2840  = blk_2840.std(0)
mean_4858 = blk_4858.mean(0)
std_4858  = blk_4858.std(0)
del blk_2840, blk_4858

# gradient on mean_2840 (edge/texture feature)
gx = cv2.Sobel(mean_2840, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(mean_2840, cv2.CV_32F, 0, 1, ksize=3)
grad_2840 = np.sqrt(gx**2 + gy**2)

# local high-pass (mean minus blurred) — sensitive to fine texture
blur = cv2.GaussianBlur(mean_2840, (0, 0), 2)
hipass_2840 = np.abs(mean_2840 - blur)

print(f'mean_2840 range: [{mean_2840.min():.0f}, {mean_2840.max():.0f}]')
print(f'std_2840  range: [{std_2840.min():.1f}, {std_2840.max():.1f}]')

## Visual inspection

Top row: full crop (downscaled). Bottom row: a zoomed sub-region showing individual ink strokes in the 2.4um output.

**What to look for**: do the bright (high-probability) spots in the warped 2.4 ink map correspond to anything visible in the 7.9um slice — fiber patterns, brightness differences, texture changes? Even a weak spatial correspondence (not perfect pixel alignment) would be meaningful.

In [ ]:
# --- full crop overview (downscaled to ~1/6 for display) ---
DS = 6
m79_ds  = cv2.resize(mean_2840, (cw // DS, ch // DS), interpolation=cv2.INTER_AREA)
ink_ds  = cv2.resize(warped_ink, (cw // DS, ch // DS), interpolation=cv2.INTER_AREA)
msk_ds  = cv2.resize(mask_crop.astype(np.float32), (cw // DS, ch // DS), interpolation=cv2.INTER_AREA)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 7.9um depth-mean
ax = axes[0]
ax.imshow(m79_ds, cmap='gray', interpolation='nearest')
ax.set_title('7.9um z28–40 mean (full crop, 6x downscaled)', fontsize=11)
ax.axis('off')

# warped 2.4um ink prob
ax = axes[1]
ax.imshow(ink_ds, cmap='hot', vmin=0, vmax=200, interpolation='nearest')
ax.set_title('2.4um ink probability (warped into 7.9 frame)', fontsize=11)
ax.axis('off')

# overlay: 7.9 as gray, 2.4 ink as red channel
ax = axes[2]
gray = np.clip(m79_ds / m79_ds[msk_ds > 0].max(), 0, 1)
alpha = np.clip(ink_ds / 180.0, 0, 1)
rgb = np.stack([gray * 0.7 + alpha * 0.8, gray * 0.7, gray * 0.7], axis=-1)
rgb = np.clip(rgb, 0, 1)
ax.imshow(rgb, interpolation='nearest')
ax.set_title('overlay: 7.9=gray, 2.4 ink=red', fontsize=11)
ax.axis('off')

plt.tight_layout()
plt.savefig('/vesuvius/_ves_tmp/corr_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved to /vesuvius/_ves_tmp/corr_overview.png')

In [ ]:
# --- zoomed view: pick three sub-regions with significant ink probability ---
# find the top-ink tile positions (highest mean warped prob)
T = 32
best = []
for y in range(0, ch - T * 5, T):
    for x in range(0, cw - T * 5, T):
        if mask_crop[y:y + T, x:x + T].mean() < 128:
            continue
        p = warped_ink[y:y + T * 5, x:x + T * 5].mean()
        best.append((p, y, x))
best.sort(reverse=True)

PATCH = T * 8   # 256px patch
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
shown = 0
seen = set()
for score, py, px in best:
    # avoid overlapping patches
    key = (py // PATCH, px // PATCH)
    if key in seen: continue
    seen.add(key)
    if shown >= 3: break
    py2 = min(py, ch - PATCH); px2 = min(px, cw - PATCH)

    p79  = mean_2840[py2:py2 + PATCH, px2:px2 + PATCH]
    p48  = mean_4858[py2:py2 + PATCH, px2:px2 + PATCH]
    pstd = std_2840[py2:py2 + PATCH, px2:px2 + PATCH]
    pink = warped_ink[py2:py2 + PATCH, px2:px2 + PATCH]

    # overlay
    g = np.clip(p79 / (p79.max() + 1e-6), 0, 1)
    a = np.clip(pink / 180.0, 0, 1)
    rgb = np.stack([g * 0.6 + a * 0.9, g * 0.6, g * 0.6], axis=-1)
    rgb = np.clip(rgb, 0, 1)

    row = shown
    axes[row][0].imshow(p79, cmap='gray'); axes[row][0].set_title(f'7.9 z28-40 @ y{py2} x{px2}')
    axes[row][1].imshow(pink, cmap='hot', vmin=0, vmax=200); axes[row][1].set_title(f'2.4um ink prob (score={score:.1f})')
    axes[row][2].imshow(rgb); axes[row][2].set_title('overlay')
    for ax in axes[row]: ax.axis('off')
    shown += 1

plt.suptitle('Zoomed patches — highest 2.4um ink probability regions, matched in 7.9um', y=1.01)
plt.tight_layout()
plt.savefig('/vesuvius/_ves_tmp/corr_zoomed.png', dpi=120, bbox_inches='tight')
plt.show()

## Tile-level correlation analysis

Aggregate to 32×32 tiles (= native 7.9um tile size). For each tile, compute:
- **mean_2840**: mean intensity z28–40 (standard ink window)
- **std_2840**: std of intensities z28–40 (texture)
- **mean_4858**: mean intensity z48–58 (alt band from depth profile)
- **std_4858**: std z48–58
- **gradient**: mean gradient magnitude (edge strength)
- **hipass**: mean high-pass (local texture, sub-pixel fiber pattern)

vs. `ink_prob`: mean warped 2.4um ink probability for that tile.

Spearman r is used because: (a) the relationship need not be linear, (b) outlier-robust.

In [ ]:
T = 32
feat_names = ['mean_2840', 'std_2840', 'mean_4858', 'std_4858', 'gradient', 'hipass']
feat_maps  = [mean_2840, std_2840, mean_4858, std_4858, grad_2840, hipass_2840]

rows = []
for y in range(0, ch - T, T):
    for x in range(0, cw - T, T):
        if mask_crop[y:y + T, x:x + T].mean() < 64:
            continue
        ip = warped_ink[y:y + T, x:x + T].mean()
        feats = [fm[y:y + T, x:x + T].mean() for fm in feat_maps]
        rows.append([y, x, ip] + feats)

rows = np.array(rows)  # (N, 2+1+6)
tile_y = rows[:, 0].astype(int)
tile_x = rows[:, 1].astype(int)
ink_arr = rows[:, 2]
feat_arr = {fn: rows[:, 3 + i] for i, fn in enumerate(feat_names)}

print(f'tiles: {len(rows)}  ink_prob range [{ink_arr.min():.1f}, {ink_arr.max():.1f}]')
print(f'ink>60 frac: {(ink_arr > 60).mean():.3f}  ink>120: {(ink_arr > 120).mean():.3f}')
print()
print(f'{"feature":<14}  {"r":>7}  {"p":>10}')
print('-' * 36)
for fn in feat_names:
    r, p = spearmanr(feat_arr[fn], ink_arr)
    print(f'{fn:<14}  {r:+.4f}  {p:.2e}')

In [ ]:
# spatial consistency: split into 4 quadrants and check sign + magnitude of best feature
# if the correlation inverts between regions it is a spatial confound, not real signal
ymid = ch // 2
xmid = cw // 2
quads = {
    'top-left' : (tile_y < ymid) & (tile_x < xmid),
    'top-right': (tile_y < ymid) & (tile_x >= xmid),
    'bot-left' : (tile_y >= ymid) & (tile_x < xmid),
    'bot-right': (tile_y >= ymid) & (tile_x >= xmid),
}

print('Spatial consistency (Spearman r per quadrant):')
print(f'{"quadrant":<12}  {"n":>5}  ' + '  '.join(f'{fn[:8]:>9}' for fn in feat_names))
print('-' * (12 + 7 + len(feat_names) * 11))
for qname, mask in quads.items():
    n = mask.sum()
    if n < 20:
        print(f'{qname:<12}  {n:>5}  (too few)')
        continue
    rs = []
    for fn in feat_names:
        r, _ = spearmanr(feat_arr[fn][mask], ink_arr[mask])
        rs.append(r)
    print(f'{qname:<12}  {n:>5}  ' + '  '.join(f'{r:+.4f}   ' for r in rs))

In [ ]:
# scatter plots: best two features vs ink_prob, colored by x-position (reveals spatial confound)
best_fns = sorted(feat_names, key=lambda fn: abs(spearmanr(feat_arr[fn], ink_arr)[0]), reverse=True)[:4]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, fn in zip(axes, best_fns):
    fv = feat_arr[fn]
    r, p = spearmanr(fv, ink_arr)
    sc = ax.scatter(fv, ink_arr, c=tile_x / cw, cmap='coolwarm', alpha=0.25, s=6)
    ax.set_xlabel(f'7.9um {fn}', fontsize=9)
    ax.set_ylabel('2.4um ink prob', fontsize=9)
    ax.set_title(f'r={r:+.3f}  p={p:.1e}', fontsize=10)
    plt.colorbar(sc, ax=ax, label='x pos (blue=left, red=right)')

plt.suptitle('Scatter: 7.9um feature vs 2.4um ink probability (color = spatial position)\n'
             'A gradient in color along the y-axis = spatial confound, not real signal',
             fontsize=10)
plt.tight_layout()
plt.savefig('/vesuvius/_ves_tmp/corr_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# extra check: does the correlation improve when restricting to tiles with HIGH ink probability?
# if the 7.9 signal only shows up where ink is dense (strong strokes), that's still useful.
print('Spearman r vs ink prob threshold (restricting to tiles WITH ink):')
print(f'{"threshold":>12}  {"n_ink":>7}  ' + '  '.join(f'{fn[:10]:>11}' for fn in feat_names))
print('-' * (14 + 9 + len(feat_names) * 13))
for thr in [30, 60, 90, 120, 150]:
    hi = ink_arr >= thr
    lo = ink_arr < 20   # definite non-ink
    mask = hi | lo
    labels = (ink_arr[mask] >= thr).astype(int)
    n = hi.sum()
    if n < 20:
        print(f'ink>={thr:>4}      {n:>7}  (too few ink tiles)')
        continue
    rs = []
    for fn in feat_names:
        r, _ = spearmanr(feat_arr[fn][mask], labels)
        rs.append(r)
    print(f'ink>={thr:>4}      {n:>7}  ' + '  '.join(f'{r:+.5f}     ' for r in rs))

## Interpretation guide

| Result | Meaning |
|---|---|
| r > +0.15, consistent across all 4 quadrants, same sign | **real weak signal** in the 7.9um data — pursue distillation or feature-engineered detector |
| r varies in sign across quadrants | **spatial confound** — ink areas happen to sit in a brighter/darker region of the sheet, not a detectable feature |
| r ≈ 0 or r < 0.05 everywhere | **no signal** — 7.9um scroll4 → scrolls 2/3 is a dead end at this energy, wait for higher-resolution scans |
| scatter plot colored gradient runs vertically | spatial confound (feature correlates with sheet position, not with ink) |
| scatter plot colored gradient runs horizontally | real signal (feature correlates with ink, regardless of spatial position) |

Note: **AUC or r ≈ 0.55–0.65** (from our earlier depth diagnostic) was a spatial confound — the ink-labeled region of the training set happened to sit in a slightly different exposure zone. The quadrant consistency check and scatter color are the decisive checks.

In [ ]:

# Display pre-computed figures (run the terminal script first if images don't exist)
from IPython.display import Image, display
import os

for fname, title in [
    ('_ves_tmp/corr_overview.png', 'Overview: 7.9um | 2.4um ink | overlay'),
    ('_ves_tmp/corr_zoomed.png',   'Zoomed: highest ink regions in 2.4um — what does 7.9um show?'),
    ('_ves_tmp/corr_scatter.png',  'Scatter: 7.9um features vs 2.4um ink probability'),
]:
    path = f'/vesuvius/{fname}'
    if os.path.exists(path):
        print(f'\n=== {title} ===')
        display(Image(filename=path, width=1200))
    else:
        print(f'Not found: {path} — run the terminal script first')
